In [13]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress

from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'


#### Beware: deleted computed_shelfmelt_AIS_IMAU_UFEMISM1_expAE05.nc in ComputedScalarsHelene have orig_... and hb_calculated_now...

In [14]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = [] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame

In [15]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]
dic_area_of_interest = {}
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Amundsen'] = ['_sector_4']
y2sec=-365.25*24*3600
factor = y2sec/10**12

In [16]:
lst = glob.glob(pth_helene+f'expAE02/shelfmelt/*.nc')
lst.sort()


In [17]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    #getting rid of old IMAU file (orig_...) and using the newly computed jb_computed... only:
    lst = lst[:-4]
    lst = lst[::-1]
    

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    
    #get shelf area
    pth_area = pth_way.replace('shelfmelt','iareafl')
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2100)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2100.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3091885461.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3091885461.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3091885461.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [18]:
df2

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,IMAU_UFEMISM4,/Volumes/CrucialX8/computing/data/antarctica/i...,1611.824341,67816.421875,8278.105469,9334.640625,140990.515625,26225.423828,219.684067,...,1646.225708,3461.944580,0.000000,228031.484375,187076.406250,5108.169434,242537.779800,94041.856976,9334.641399,420216.031250
1,expAE02,IMAU_UFEMISM3,/Volumes/CrucialX8/computing/data/antarctica/i...,1891.794556,74746.570312,8521.494141,9050.348633,151186.796875,29280.572266,597.917725,...,1816.268311,4224.842285,4.919946,245397.062500,201505.343750,6046.030273,253945.894875,104027.149368,9050.347588,452948.437500
2,expAE02,IMAU_UFEMISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,880.176331,73964.648438,3388.902100,3482.702148,144775.937500,28755.875000,195.901077,...,1432.764282,2647.808594,23.858397,226492.375000,177799.343750,4104.431641,247906.557846,102720.519241,3482.701520,408396.218750
3,expAE02,IMAU_UFEMISM1,/Volumes/CrucialX8/computing/data/antarctica/i...,974.645081,71335.101562,4468.673828,2365.238770,129208.289062,25799.421875,-0.000000,...,1411.865112,2671.535400,-0.000000,208351.921875,164163.296875,4083.401123,226168.220894,97134.514526,2365.238376,376598.718750
4,expAE02,VUW_PISM2_s4,/Volumes/CrucialX8/computing/data/antarctica/i...,11748.368164,62546.175781,17854.658203,21344.078125,211816.953125,37506.578125,6693.994629,...,2835.304688,5330.658203,14464.389648,325310.218750,360697.187500,22630.353516,373485.112577,100052.737001,21344.079353,708637.687500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,expAE05,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,9006.089844,47834.742188,23822.015625,23822.710938,2710.322510,17403.414062,3779.043945,...,2500.085449,4152.203613,14909.415039,107195.890625,115427.257812,21561.705078,6885.203062,65238.163511,23822.707062,244184.859375
168,expAE05,DOE_MALI_8km_AntMean,/Volumes/CrucialX8/computing/data/antarctica/i...,9125.056641,55166.757812,23747.599609,29895.007812,9863.629883,18904.748047,2997.808105,...,227.771896,1344.373657,10883.929688,127798.078125,68881.773438,12456.077148,17337.025363,74071.517281,29895.011215,209135.937500
169,expAE05,DOE_MALI_8km_Ant95,/Volumes/CrucialX8/computing/data/antarctica/i...,9540.550781,63995.343750,24670.841797,32650.589844,10618.098633,21449.373047,3449.718994,...,268.997559,1334.802368,10738.963867,141475.421875,75720.351562,12342.761719,19004.323269,85444.713175,32650.591217,229538.562500
170,expAE05,DOE_MALI_4km,/Volumes/CrucialX8/computing/data/antarctica/i...,9402.107422,56845.140625,24807.822266,31764.996094,9940.785156,19144.416016,4943.729004,...,218.680893,1539.677490,11594.932617,132760.875000,73897.468750,13353.287109,17472.244884,75989.562393,31764.998978,220011.656250


In [19]:
# avg and abs 2200
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()    
    #getting rid of old IMAU file (orig_...) and using the newly computed jb_computed... only:
    lst = lst[:-4]
    lst = lst[::-1]
    

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2200)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2200.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3255398805.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3255398805.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3255398805.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [25]:
# avg and abs 2300
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    #getting rid of old IMAU file (orig_...) and using the newly computed jb_computed... only:
    lst = lst[:-4]
    lst = lst[::-1]

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    
    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where(d.time.values <= 2300)
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d.shelfmelt.values[:tmin]* factor/d2.iareafl.values[:tmin] #T/m^3 yr
    yBMB =d.shelfmelt.values[:tmin]* factor
    
   
    yais =y[ind].mean()
    yaisBMB =yBMB[ind].cumsum()[-1]
    
    
    df['AIS'].iloc[i]=  yais
    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        y =d['shelfmelt_sector_'+str(j)].values[:tmin]* factor/d2['iareafl_sector_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_sector_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        y =d['shelfmelt_region_'+str(j)].values[:tmin]* factor/d2['iareafl_region_'+str(j)].values[:tmin]  #GT/m^3 yr
        yBMB=d['shelfmelt_region_'+str(j)].values[:tmin]* factor
        
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        ysa = np.zeros((len(sectors),tmin))
        
        areas = np.zeros((len(sectors),tmin))
    
        for j,sector in enumerate(sectors):
            ys[j,:] = d[f'shelfmelt{sector}'].values[:tmin].copy()* factor
            areas[j,:] = d2[f'iareafl{sector}'].values[:tmin].copy()
            ysa[j,:]=ys[j,:]*areas[j,:]
        yBMB =  np.nansum(ys,axis = 0)
        y =  np.nansum(ys,axis = 0)/np.nansum(areas,axis= 0)
        df[key].iloc[i] = y[ind].mean()
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_shelfmelt_2300.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3495047340.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3495047340.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_66386/3495047340.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [21]:
df_test= df2[df2.Experiment == 'expAE02']

In [23]:
df_test

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,IMAU_UFEMISM4,/Volumes/CrucialX8/computing/data/antarctica/i...,2745.033936,162937.468750,8291.433594,9929.442383,241587.953125,40228.941406,898.381775,...,1673.774658,3485.093506,0.000000,4.254914e+05,242475.640625,5158.866699,371857.825321,203166.458339,9929.441125,6.731260e+05
1,expAE02,IMAU_UFEMISM3,/Volumes/CrucialX8/computing/data/antarctica/i...,3078.729736,179971.656250,8531.175781,10589.115234,270359.406250,45792.753906,1250.393066,...,1839.054443,4236.963379,4.919946,4.725306e+05,278840.562500,6080.937500,418033.716595,225764.475348,10589.106896,7.574512e+05
2,expAE02,IMAU_UFEMISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,880.176331,141574.093750,3388.902100,3490.639404,201610.578125,44358.929688,195.901077,...,1432.764282,2647.808594,23.858397,3.509443e+05,219460.906250,4104.431641,326852.024283,185933.005062,3490.638750,5.745096e+05
3,expAE02,IMAU_UFEMISM1,/Volumes/CrucialX8/computing/data/antarctica/i...,980.030762,122743.648438,4468.673828,2468.053955,177003.093750,37955.757812,-0.000000,...,1411.865112,2671.535400,430.405670,3.076633e+05,197946.250000,4513.806641,292486.539061,160699.433270,2468.053848,5.101237e+05
4,expAE02,VUW_PISM2_s4,/Volumes/CrucialX8/computing/data/antarctica/i...,58893.988281,272715.031250,78759.843750,301957.468750,654504.750000,48881.550781,35832.753906,...,4601.173340,7815.533691,64841.601562,1.366830e+06,830648.875000,77258.265625,941399.503599,321596.582266,301957.467262,2.274738e+06
5,expAE02,VUW_PISM2_s3,/Volumes/CrucialX8/computing/data/antarctica/i...,48071.070312,248205.578125,74845.484375,105287.312500,622808.687500,48590.250000,32538.830078,...,4799.101562,7825.639160,62899.832031,1.099218e+06,776325.687500,75524.601562,888369.582457,296795.858001,105287.318459,1.951068e+06
6,expAE02,VUW_PISM2_s2,/Volumes/CrucialX8/computing/data/antarctica/i...,56130.468750,246540.781250,73023.664062,106060.804688,628941.437500,49213.351562,32578.037109,...,4688.640137,8346.356445,64616.292969,1.110696e+06,752164.437500,77651.289062,897561.727003,295754.152902,106060.759361,1.940512e+06
7,expAE02,VUW_PISM2_s1,/Volumes/CrucialX8/computing/data/antarctica/i...,58103.027344,296220.531250,80455.484375,173632.203125,675637.875000,49289.648438,45024.835938,...,4395.391113,7557.734375,64985.996094,1.284050e+06,882118.562500,76939.109375,970062.271370,345510.211594,173632.240677,2.243106e+06
8,expAE02,VUW_PISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,57438.007812,267509.343750,75581.710938,183229.843750,648727.562500,49058.437500,33633.093750,...,4608.660645,8144.552734,65088.472656,1.232486e+06,811350.312500,77841.718750,930157.855059,316567.732082,183229.849823,2.121679e+06
9,expAE02,VUW_PISM1_s4,/Volumes/CrucialX8/computing/data/antarctica/i...,49242.960938,235349.968750,78229.164062,95204.492188,566876.562500,47083.429688,34308.898438,...,3704.051270,8837.822266,59684.265625,1.024903e+06,741846.187500,72226.132812,818521.089092,282433.414315,95204.519936,1.838975e+06
